In [ ]:
import cv2

# Initialize camera
cap = cv2.VideoCapture(0)

# Fetch frame properties
frame_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
frame_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps = int(cap.get(cv2.CAP_PROP_FPS)) or 30  # Fallback if metadata reads 0

# Set up video writer
fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out = cv2.VideoWriter('output_video.mp4', fourcc, fps, (frame_width, frame_height))

# State counters and tracking flags
img_counter = 0
is_recording = False  # Track whether we are actively writing to the video file

print("Camera stream initialized...")

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        print("Error: Failed to grab frame.")
        break

    # Dynamic status bar based on recording state
    if is_recording:
        info_text = "STATUS: RECORDING | 'v' to Pause | 'c' to Capture | 'q' to Quit"
        text_color = (0, 0, 255)  # Red text for active recording
    else:
        info_text = "STATUS: PAUSED | 'v' to Record | 'c' to Capture | 'q' to Quit"
        text_color = (0, 255, 0)  # Green text for standby mode

    # Create a clean display frame copy so text overlays do not get saved into your output video/images
    display_frame = frame.copy()
    cv2.putText(display_frame, info_text, (15, 40), cv2.FONT_HERSHEY_SIMPLEX, 0.6, text_color, 2)
        
    # Write raw frame to output video file only if recording is toggled ON
    if is_recording:
        out.write(frame)

    # Display the frame with instructions
    cv2.imshow('Camera System', display_frame)
    
    # Capture keyboard input (MOVED UP to prevent NameError execution crashes)
    key = cv2.waitKey(1) & 0xFF
    
    # Press 'v' to toggle video recording on/off
    if key == ord('v'):
        is_recording = not is_recording
        state_str = "STARTED" if is_recording else "PAUSED"
        print(f"Video recording {state_str}.")
    
    # Press 'c' to take a clean snapshot
    elif key == ord('c'):
        filename = f"captured_img_{img_counter}.jpg"
        cv2.imwrite(filename, frame)  # Saves clean frame without text overlay
        print(f"Saved snapshot: {filename}")
        img_counter += 1
        
    # Press 'q' to completely exit application
    elif key == ord('q'):
        print("Closing application.")
        break

# Release resources cleanly
cap.release()
out.release()
cv2.destroyAllWindows()
